# Acceptance: picking NUTS settings from pilot runs

**This is a sign-off page, not a tutorial.** It exercises roadmap step 47
(composability package `CP7`, findings `CX36`–`CX38`). A new calibration needs
NUTS settings before its first real run: diagonal or dense mass matrix, where the
metric adaptation starts, `max_tree_depth`, `target_accept_prob`, initial step
size and warm-up length. `wf.NutsTuning` runs short, identically seeded pilots of
candidate settings from the multi-start optima and ranks them by the cost of
reaching a target number of effective draws, with warm-up cost reported
separately from sampling speed. Nothing here asserts; the automated checks live
in `tests/test_calib_workflow_tune.py::test_gate_*`.

The posteriors are notebook 28's: the **joint** model (an SIR plus six partially
pooled historical outbreaks: a plate, a non-centred latent, 10 dimensions) and
the **main-only** SIR, whose data leave a ridge between infection and recovery.
Run every cell top to bottom (about three minutes). Each section ends with a
**What to check** box.

| | You are checking that… |
|---|---|
| 1 | the Laplace metric at the best optimum shows the pooling correlations, with no clipped directions |
| 2 | one log density + gradient is timed in tens of microseconds, far below the pilots' wall time |
| 3 | starting the metric from Laplace cuts the leapfrog steps per warm-up iteration by more than half, diagonal and dense alike |
| 4 | stage B varies one trajectory setting at a time around the stage-A winner |
| 5 | the ranking depends on the target ESS: cost = warm-up + target / speed, and the curves cross |
| 6 | on the main-only ridge, the better dense metric gives at least twice diagonal's effective draws per gradient; a fixed Laplace metric fails |
| 7 | the recommended factory converges on every element (R-hat ≤ 1.01, ESS ≥ 400) |

In [ ]:
import warnings

import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
import optax
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import BayesianModel, NegativeBinomial, NumpyroPosterior, Uniform
from summer4.epi.calibration import workflow as wf

## The two posteriors

Exactly notebook 28's models and data (same seeds). `bm` is the main-only SIR;
`joint_post` adds the pooled outbreak durations, with
`recovery = exp(mu + tau * z_main)`.

In [ ]:
# --- notebook 28's settings (kept in sync with tests/gate_custom_posterior.py) ---
DATA_SEED = 7
K_OUTBREAKS = 6
N_DUR = 30
TRUE_INFECTION = 0.45
TRUE_MU = -1.1
TRUE_TAU = 0.1
MU_PRIOR_LOC = float(np.log(0.25))
MU_PRIOR_SCALE = 1.0
TAU_PRIOR_SCALE = 0.15
NB_DISPERSION = 20.0
POP_SIZE = 1.0e6

rng = np.random.default_rng(DATA_SEED)
z_hist = rng.normal(size=K_OUTBREAKS)
z_main = float(rng.normal())
TRUE_RECOVERY = float(np.exp(TRUE_MU + TRUE_TAU * z_main))
rates_hist = np.exp(TRUE_MU + TRUE_TAU * z_hist)
durations = rng.exponential(1.0 / rates_hist[:, None], size=(K_OUTBREAKS, N_DUR))

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)
flow_model = FlowModel(pmap)
flow_model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=MixingMatrix(pop, np.array([[1.0]])),
            kind=FOIKind.FREQUENCY,
            contact_rate=Param("infection"),
        ),
    )
)
flow_model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
cm = flow_model.compile()

y0_values = np.zeros(pmap.size)
y0_values[pmap.select(state["S"])] = POP_SIZE - 20.0
y0_values[pmap.select(state["I"])] = 20.0
y0 = PropertyData.wrap(pmap, y0_values)
infectious = Compartments(where=state["I"])
run_kwargs = {"t0": 0.0, "t1": 180.0, "dt": 0.5, "solver": "euler"}
times = np.arange(0.0, 21.0, 2.0)

clean = cm.run(
    {"infection": TRUE_INFECTION, "recovery": TRUE_RECOVERY},
    y0,
    save=SavePlan(requests={"I": SaveRequest(infectious, ts=times)}),
    **run_kwargs,
)
expected = np.asarray(clean["I"].values.data).reshape(-1)
observed = rng.negative_binomial(20, 20 / (20 + expected)).astype(float)

bm = BayesianModel(
    cm,
    {},
    priors=(Uniform("infection", 0.1, 1.5), Uniform("recovery", 0.02, 1.0)),
    targets=TargetSet(
        targets=(
            Target(
                key="I",
                times=times,
                values=observed,
                quantity=infectious,
                likelihood=NegativeBinomial(dispersion=NB_DISPERSION),
            ),
        )
    ),
    y0=y0,
    run_kwargs=run_kwargs,
)

truth = {
    "infection": TRUE_INFECTION,
    "recovery": TRUE_RECOVERY,
    "mu": TRUE_MU,
    "tau": TRUE_TAU,
    "z_main": z_main,
}


def joint_model() -> None:
    mu = numpyro.sample("mu", dist.Normal(MU_PRIOR_LOC, MU_PRIOR_SCALE))
    tau = numpyro.sample("tau", dist.HalfNormal(TAU_PRIOR_SCALE))
    infection = numpyro.sample("infection", dist.Uniform(0.1, 1.5))
    with numpyro.plate("outbreaks", K_OUTBREAKS):
        z = numpyro.sample("z", dist.Normal(0.0, 1.0))
        rate = jnp.exp(mu + tau * z)
        with numpyro.plate("dur", N_DUR):
            numpyro.sample(
                "duration",
                dist.Exponential(rate),
                obs=jnp.asarray(durations.T),
            )
    z_main_s = numpyro.sample("z_main", dist.Normal(0.0, 1.0))
    recovery = jnp.exp(mu + tau * z_main_s)
    numpyro.factor(
        "main",
        bm.log_likelihood({"infection": infection, "recovery": recovery}),
    )


joint_post = NumpyroPosterior(joint_model)

## Starting points: multi-start optima

Every pilot starts from the same `Candidates`: Latin hypercube, keep the best
four, run Adam, and score the end points. The Laplace metric and the gradient
timing use the best of them.

In [ ]:
# --- tuning settings (kept in sync with tests/gate_nuts_tuning.py) ---
LHS_M = 48
LHS_SEED = 0
N_STARTS = 4
OPT_LR = 0.05
OPT_CHUNK = 50
OPT_MAX_STEPS = 800
OPT_SEED = 0
NUM_CHAINS = 4
PILOT_SAMPLES = 150
NUM_WARMUP = 100
TARGET_ESS = 400.0
TUNE_SEED = 0
RUN_SAMPLES = 250
RUN_SEED = 1
RUN_MAX_SAMPLES = 2000


def optimised_starts(post):
    design = wf.evaluate(post, wf.lhs(post, LHS_M, seed=LHS_SEED), batch_size=16)
    opt = wf.optimize(
        post,
        design.best(N_STARTS),
        method=wf.Optax(optax.adam(OPT_LR)),
        chunk_steps=OPT_CHUNK,
        max_steps=OPT_MAX_STEPS,
        seed=OPT_SEED,
    )
    return wf.evaluate(post, opt.candidates, batch_size=N_STARTS)


starts_joint = optimised_starts(joint_post)
starts_main = optimised_starts(bm)
pd.DataFrame(
    {
        "log density": np.asarray(starts_joint.log_density),
        **{site: np.asarray(starts_joint.params[site]) for site in ("infection", "mu", "tau", "z_main")},
    }
).round(3)

## 1. The Laplace metric

`wf.laplace_metric` takes the inverse negative Hessian of the log density at the
best optimum, in unconstrained space, with eigenvalues clipped to
`[min_var, max_var]`. Its rows follow numpyro's mass-matrix block order: sites
sorted by name, plate sites raveled (`z[0]` … `z[5]`). The heat map shows it as
a correlation matrix.

In [ ]:
metric = wf.laplace_metric(joint_post, starts_joint)
names = list(metric.element_names)
corr = metric.covariance / np.outer(metric.std, metric.std)

fig = px.imshow(
    pd.DataFrame(corr, index=names, columns=names),
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    text_auto=".2f",
    title="1. Laplace correlation at the best optimum (unconstrained, numpyro block order)",
)
fig.update_layout(height=600, xaxis_title="element", yaxis_title="element")
fig.show()

pd.Series(
    {
        "elements": len(names),
        "clipped directions": metric.clipped,
        "smallest eigenvalue of -H": float(metric.eigenvalues.min()),
        "largest eigenvalue of -H": float(metric.eigenvalues.max()),
        "corr(infection, z_main)": float(corr[names.index("infection"), names.index("z_main")]),
        "corr(mu, z[0])": float(corr[names.index("mu"), names.index("z[0]")]),
    },
    name="1",
)

> **What to check**
>
> - Rows and columns read `infection`, `mu`, `tau`, `z[0]` … `z[5]`, `z_main` (sorted names).
> - `infection` and `z_main` are strongly correlated (about **0.85**): the main
>   data pin `infection − recovery`, and `z_main` sets `recovery`.
> - `mu` is negatively correlated with every `z[j]` (about **−0.4**): the
>   non-centred pooling trade-off.
> - `clipped directions` is **0** and the smallest eigenvalue is positive.

## 2. What one gradient costs

`wf.time_gradient` times one jitted log density plus gradient, after a warm
call, blocking on the result each time (JAX dispatch is asynchronous, so an
unblocked timer measures only the dispatch).

In [ ]:
grad_seconds = pd.Series(
    {
        "joint": wf.time_gradient(joint_post, starts_joint),
        "main-only": wf.time_gradient(bm, starts_main),
    },
    name="seconds per log density + gradient",
)
fig = (grad_seconds * 1e6).plot(
    kind="bar", title="2. One log density + gradient (microseconds, median of 20)"
)
fig.update_layout(xaxis_title="posterior", yaxis_title="microseconds", showlegend=False)
fig.show()
grad_seconds

> **What to check**
>
> - Both bars are tens of microseconds. At this cost, a pilot's wall time is
>   mostly JIT compilation (each warm-up round and each sampling chunk
>   compiles), so the
>   tables below report gradient evaluations as well as seconds. A model whose
>   gradient takes milliseconds (tbcore NH4: 4.4 ms) is dominated by gradients
>   instead.

## 3. Stage A: how to start and adapt the mass matrix

`wf.metric_screen()` holds numpyro's other defaults (tree depth 10, target
acceptance 0.8) and varies the metric: `diag`, `diag+laplace` (diagonal, started
from the Laplace variances), `dense` (from the identity), `dense+laplace`, and
`laplace fixed` (dense Laplace metric, never adapted). Each pilot runs
`wf.warmup_until` (doubling rounds; the next round starts from the previous
round's adapted metric), then two chunks of `PILOT_SAMPLES` draws per chain.
Speed is timed on the second chunk (which still recompiles numpyro's outer
loop, see `futureplans/mcmc-chunk-recompile.md`).

In [ ]:
tuning = wf.NutsTuning(
    joint_post,
    starts_joint,
    num_chains=NUM_CHAINS,
    pilot_samples=PILOT_SAMPLES,
    seed=TUNE_SEED,
    on_pilot=lambda row: print(f"{row['label']:>32}: ESS per gradient {row.get('ess_per_grad')}"),
)
tuning.run(wf.metric_screen(NUM_WARMUP))
stage_a = tuning.table(TARGET_ESS)

steps = stage_a[["warmup_steps_mean", "num_steps_mean"]].rename(
    columns={
        "warmup_steps_mean": "per warm-up iteration",
        "num_steps_mean": "per draw after warm-up",
    }
)
fig = steps.plot(
    kind="bar",
    barmode="group",
    title="3. Joint posterior: leapfrog steps (gradient evaluations) per iteration",
)
fig.update_layout(xaxis_title="setting", yaxis_title="leapfrog steps", legend_title="")
fig.show()

stage_a[
    [
        "rank",
        "warmup_rounds",
        "num_warmup",
        "warmup_grad_evals",
        "warmup_seconds",
        "ess_min",
        "cost_grads",
        "ess_per_grad",
        "ess_per_second",
        "divergence_frac",
        "rhat_max",
        "why",
    ]
].round(4)

> **What to check**
>
> - Blue bars (warm-up): `diag+laplace` and `dense+laplace` take well under half
>   the steps per warm-up iteration of `diag` and `dense` (about **25 vs 80**
>   and **18 vs 80**). From the identity, early warm-up integrates with the
>   wrong scales and builds long trajectories: tbcore NH4's 54-minute dense
>   warm-up. The Laplace start removes most of that.
> - Red bars (sampling) are much closer together: once adapted, the metrics
>   end up similar on this posterior.
> - In the table, `why` names any pilot that is not usable (warm-up failed,
>   divergences above 1%, or a worst-element R-hat above 1.1). Differences in
>   `ess_per_grad` under about 30% are within pilot noise.

## 4. Stage B: trajectory settings around the winner

`wf.trajectory_screen(best)` changes one thing at a time around stage A's best
setting (rank 1 above: lowest gradient cost to `TARGET_ESS`): `max_tree_depth` 6 and 8, `target_accept_prob`
0.9 and 0.95, and `find_heuristic_step_size=True` (searching for the initial
step size). Any other NUTS option can be a `wf.NutsSetting` of its own.

In [ ]:
base = tuning.best(TARGET_ESS).setting
tuning.run(wf.trajectory_screen(base))
table = tuning.table(TARGET_ESS)
stage_b = table.loc[[base.label] + [p.setting.label for p in tuning.pilots[5:]]]

long = stage_b[["num_steps_mean", "step_size", "accept_mean", "treedepth_frac"]].reset_index()
long = long.melt(id_vars="label", var_name="statistic", value_name="value")
fig = px.bar(
    long,
    x="label",
    y="value",
    facet_row="statistic",
    title=f"4. Stage B around '{base.label}'",
    height=750,
)
fig.update_yaxes(matches=None, title="")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(xaxis_title="setting")
fig.show()

stage_b[["rank", "num_steps_mean", "num_steps_max", "treedepth_frac", "ess_per_grad", "why"]]

> **What to check**
>
> - Every stage-B label starts with the stage-A winner's label.
> - Raising the target acceptance shrinks the step size and lengthens
>   trajectories (`num_steps_mean` up).
> - `treedepth_frac` (draws that hit `max_tree_depth`) is near zero at depth 8
>   and 10: this posterior never needs deep trees, so lowering the depth can
>   only matter where it binds.

## 5. Cost depends on how many draws you want

For each usable pilot, `cost = warm-up + target ESS / speed`, counted in
gradient evaluations (`cost_grads`, the default ranking: reproducible for a
fixed seed and the same on any hardware) or in this machine's seconds
(`cost_seconds`, which also counts compilation and per-step overhead, plus timer
noise). A setting with a cheap warm-up wins for a few hundred draws; the
fastest sampler wins for many thousands.

In [ ]:
targets = np.geomspace(30, 30_000, 40)
usable = table[table["usable"]]
curves = pd.DataFrame(
    {
        label: row["warmup_grad_evals"] + targets / row["ess_per_grad"]
        for label, row in usable.iterrows()
    },
    index=pd.Index(targets, name="target ESS"),
)
fig = curves.plot(
    title="5. Gradient evaluations to reach a target ESS (warm-up + sampling)",
    log_x=True,
    log_y=True,
)
fig.add_vline(x=TARGET_ESS, line_dash="dash", line_color="gray")
fig.update_layout(
    xaxis_title="target ESS", yaxis_title="gradient evaluations", legend_title="setting"
)
fig.show()

pd.DataFrame(
    {
        "best at 50": tuning.best(50).setting.label,
        f"best at {TARGET_ESS:g}": tuning.best(TARGET_ESS).setting.label,
        "best at 20000": tuning.best(20_000).setting.label,
        f"best at {TARGET_ESS:g}, by seconds": tuning.best(TARGET_ESS, by="cost_seconds").setting.label,
    },
    index=["setting"],
).T

> **What to check**
>
> - Each curve is flat at its warm-up cost for small targets and then rises
>   with slope `1 / ess_per_grad`; the lowest curve at the dashed line is
>   rank 1.
> - The `dense+laplace` curve starts higher (its warm-up needed a second
>   round) but rises more slowly, and crosses below `diag+laplace` at a few
>   thousand effective draws: `best at 20000` names a different setting from
>   `best at 50`. That is the NH4 trade-off made explicit.
> - `best at 400, by seconds` may differ from rank 1: at tens of microseconds
>   per gradient, wall time is mostly compilation, overhead and timer noise.

## 6. The main-only ridge: stage A

On a two-parameter posterior that is almost a line, a diagonal metric can only
take steps along the axes, so it needs many short steps per independent draw;
any dense metric aligns with the ridge. The ridge is curved, so a **fixed**
Gaussian metric cannot follow it everywhere.

In [ ]:
main_tuning = wf.NutsTuning(
    bm, starts_main, num_chains=NUM_CHAINS, pilot_samples=PILOT_SAMPLES, seed=TUNE_SEED
).run(wf.metric_screen(NUM_WARMUP))
main_table = main_tuning.table(TARGET_ESS)

fig = main_table["ess_per_grad"].plot(
    kind="bar", title="6. Main-only ridge: effective draws per gradient evaluation"
)
fig.update_layout(xaxis_title="setting", yaxis_title="ESS per gradient", showlegend=False)
fig.show()
main_table[["rank", "num_warmup", "num_steps_mean", "ess_min", "ess_per_grad", "rhat_max", "why"]]

> **What to check**
>
> - The taller of the `dense` and `dense+laplace` bars is at least **2×** the
>   `diag` bar. Which dense bar is taller varies between runs and JAX versions:
>   pilot ESS from 600 draws on a ridge is noisy, so read the bars as rough
>   guides, not precise rates.
> - `laplace fixed` is marked `R-hat` in `why`: a fixed Gaussian metric at one
>   point of a curved ridge does not mix. Adapting from the Laplace start
>   (`dense+laplace`) does.

## 7. Use the recommendation

`tuning.recommended()` is the winning setting with `num_warmup` set to the
length its warm-up passed at; `tuning.make_mcmc(...)` is its factory, with the
Laplace metric if the setting uses one. The run below is the ordinary staged
workflow: `warmup_until` with the pilots' warm-up check `wf.PILOT_WARMUP_RULE` (so the
recommended length passes again), logging each round as it lands; then
`sample_until` with `StopRule(per_element=True)`, which judges R-hat and ESS on
the worst element of the plate site `z` rather than its average, and keeps
`num_steps` and `accept_prob` for every draw.

In [ ]:
recommended = tuning.recommended(TARGET_ESS)
print("recommended:", recommended)
warm = wf.warmup_until(
    tuning.make_mcmc(num_samples=RUN_SAMPLES, target_ess=TARGET_ESS),
    recommended.num_warmup,
    wf.PILOT_WARMUP_RULE,
    init_params=starts_joint.init_params(NUM_CHAINS, seed=RUN_SEED),
    carry_metric=True,
    seed=RUN_SEED,
    on_round=lambda row: print(
        f"warm-up round {row['round']}: {row['num_warmup']} iterations, "
        f"{row['grad_evals']} gradients, failed: {row['failed']}"
    ),
)
run = wf.sample_until(
    warm.mcmc,
    wf.StopRule(
        rhat=1.01,
        ess=TARGET_ESS,
        per_element=True,
        max_divergence_frac=0.01,
        max_samples=RUN_MAX_SAMPLES,
    ),
    seed=RUN_SEED,
    extra_fields=("num_steps", "accept_prob"),
)

# Laplace scales against the posterior's, both in unconstrained space.
draws = {k: np.asarray(v) for k, v in run.samples.items()}
n = NUM_CHAINS * draws["mu"].shape[1]
z = joint_post.unconstrain({k: v.reshape(n, *v.shape[2:]) for k, v in draws.items()})
posterior_std = np.concatenate([np.asarray(z[s]).reshape(n, -1) for s in metric.sites], axis=1).std(0)
scales = pd.DataFrame({"Laplace": metric.std, "posterior": posterior_std}, index=names)
fig = scales.plot(
    kind="scatter",
    x="posterior",
    y="Laplace",
    text=scales.index,
    title="7. Marginal scales: Laplace metric vs posterior draws (unconstrained)",
)
lim = [0, float(scales.max().max()) * 1.1]
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line={"dash": "dash"}, name="y = x"))
fig.update_traces(textposition="top center", selector={"mode": "markers+text"})
fig.update_layout(xaxis_title="posterior std", yaxis_title="Laplace std", height=520)
fig.show()

display_cols = ["samples", "rhat_max", "rhat_element_max", "ess_bulk_min", "ess_bulk_element_min"]
print(run.progress[display_cols + ["decision"]])
run.diagnostics.round(4)

> **What to check**
>
> - The warm-up log printed one line per round as it finished. With a new
>   seed the recommended length is a starting guess: here a second round was
>   needed (round 1 failed on R-hat), which `warmup_until` handles.
> - `run.progress` ends with `decision` **diagnostics**: worst-element R-hat
>   ≤ 1.01 and worst-element bulk ESS ≥ 400.
> - In `run.diagnostics`, the plate site `z` has `rhat_element_max` ≥ `rhat`
>   and `ess_bulk_element_min` ≤ `ess_bulk`: the mean over its six elements is
>   what `StopRule` used to see.
> - Scale points lie near the dashed line, within a factor of three: the
>   Laplace metric is a good *start*, not the posterior. `tau` sits below the
>   line (the optimum is near `tau`'s funnel neck, where curvature is high).

## Rerunning as the model changes

The whole tuning is

```python
tuning = wf.NutsTuning(post, starts, num_chains=4, pilot_samples=150)
tuning.run(wf.metric_screen())
tuning.run(wf.trajectory_screen(tuning.best(target_ess).setting))
make_mcmc = tuning.make_mcmc(num_samples=500, target_ess=target_ess)
```

which is exactly `wf.tune_nuts(post, starts, target_ess=...)` (tested). After a
model change, rerun it on the new optima; to skip the search, reuse
`tuning.recommended()` and recompute only the Laplace metric. The best pilot's
chains are already warmed up: `tuning.best().run.extend(stop)` keeps sampling
them without a new warm-up.